# MCP Security Governance and Risk Management

This lab turns a realistic cross-tenant MCP failure scenario into an evidence-backed, versioned risk decision. It is offline and deterministic: no production system, identity provider, or GRC platform is contacted.

## Learning contract

You will distinguish inherent from residual risk, verify signed and scoped evidence, expose control gaps, authorize an exact decision, exercise a bounded exception, invalidate stale authorization after material change, and compute portfolio metrics with visible denominators.

In [ ]:
from dataclasses import replace
from pathlib import Path
from pprint import pprint
import runpy

lab = runpy.run_path(Path('lab.py'))
NOW = 1_000
principals, controls, authority, engine, approvals, registry, record = lab['demo_components'](NOW)
evidence = lab['evidence_fixture'](authority, record, NOW)
print(f'Loaded {len(controls)} controls and {len(evidence)} signed evidence artifacts.')

## 1. Frame a causal risk scenario

The support MCP server serves multiple tenants. The business objective is legitimate, but a confused or compromised server could use a valid tool call against another tenant. A useful risk record says what causes what consequence; it is not just a label such as *data leakage*.

In [ ]:
pprint({
    'business_objective': record.business_objective,
    'threat_event': record.threat_event,
    'vulnerability': record.vulnerability,
    'consequence': record.consequence,
    'assets': record.affected_assets,
    'tenant_scope': record.tenant_scope,
    'system_version': record.system_version,
})

## 2. Apply a declared ordinal matrix

The matrix produces decision categories, not money and not probability. Multiplying ordinal labels does **not** create expected loss, and averaging these categories across a portfolio would create false precision.

In [ ]:
risk_band = lab['risk_band']
inherent = risk_band(record.inherent_likelihood, record.inherent_impact)
residual = risk_band(record.residual.likelihood, record.residual.impact)
print({'inherent': inherent.value, 'residual': residual.value})
assert inherent.value == 'high'
assert residual.value == 'moderate'

## 3. Translate controls into testable claims

A control name is not evidence. Each control declares an objective, an accountable owner, the evidence kinds required to support it, and mappings to external frameworks.

In [ ]:
for control in controls:
    pprint({
        'control': control.control_id,
        'objective': control.objective,
        'required_evidence': sorted(kind.value for kind in control.required_evidence),
        'framework_refs': control.framework_refs,
    })

## 4. Verify evidence as a scoped security object

The lab uses `cryptography` Ed25519 signatures. Verification also binds the claim to the system, tenant, immutable subject version, trusted producer, validity window, outcome, population, numerator, denominator, and unit.

In [ ]:
verified = [engine.validate_evidence(item, record, NOW + 1) for item in evidence]
for item in verified:
    print(item.evidence_id, item.kind.value, f'{item.numerator}/{item.denominator}', item.unit)

### Adversarial check: a green-looking artifact is not automatically trustworthy

Changing one signed field must make the artifact unusable. The engine fails closed and emits a stable reason code.

In [ ]:
tampered = replace(evidence[0], artifact=replace(evidence[0].artifact, numerator=39))
try:
    engine.validate_evidence(tampered, record, NOW + 1)
    raise AssertionError('tampering should not verify')
except lab['GovernanceError'] as exc:
    print(exc.code)
    assert exc.code == 'EVIDENCE_SIGNATURE_INVALID'

In [ ]:
wrong_version_artifact = replace(evidence[0].artifact, subject_version='support-v6')
wrong_version = authority.sign(wrong_version_artifact)
try:
    engine.validate_evidence(wrong_version, record, NOW + 1)
    raise AssertionError('evidence for another version should not verify')
except lab['GovernanceError'] as exc:
    print(exc.code)
    assert exc.code == 'EVIDENCE_SUBJECT_MISMATCH'

## 5. Expose missing proof instead of laundering uncertainty

Remove recovery-drill evidence. The result must name both the unsupported residual assessment and the control-specific evidence gap.

In [ ]:
gap = engine.evaluate(record, evidence[:2], (), NOW + 1)
pprint(gap.blockers)
assert 'residual_assessment_evidence_missing' in gap.blockers
assert any('CTRL-RECOVERY' in blocker for blocker in gap.blockers)

## 6. Propose and authorize an exact decision

A proposal is policy-derived and short-lived. A separate risk manager signs its exact digest. The receipt is role-aware, version-bound, expiring, and atomically single-use.

In [ ]:
proposal = engine.propose_decision(record, evidence, (), NOW + 1, 'DEC-NB-001')
pprint(proposal.payload())
assert proposal.recommendation is lab['Decision'].APPROVE
assert not proposal.blockers

In [ ]:
receipt = approvals.issue(
    'risk_decision', proposal.payload(), record.owner_id,
    'approver:risk', 'risk-manager', NOW + 1,
)
authorized = registry.apply_decision(
    proposal, receipt.receipt_id, record.owner_id, 'op-notebook-001', NOW + 2,
)
print(authorized.state.value, authorized.version, authorized.decision_id)
assert authorized.state is lab['RiskState'].AUTHORIZED

## 7. Exercise a bounded control exception

An exception temporarily waives one control requirement; it does not accept the whole risk. It must identify scope, owner, justification, compensation, supporting evidence, review time, expiry, immutable version, and CISO approval.

In [ ]:
_, _, authority2, engine2, approvals2, registry2, record2 = lab['demo_components'](NOW)
record2 = replace(
    record2,
    residual=replace(record2.residual, evidence_ids=('EV-AUTHZ-TEST', 'EV-AUTHZ-TELEMETRY')),
)
registry2.records[record2.risk_id] = record2
evidence2 = lab['evidence_fixture'](authority2, record2, NOW)[:2]

In [ ]:
request = lab['ExceptionRequest'](
    'EX-NB-001', record2.risk_id, record2.system_id, record2.tenant_scope,
    'CTRL-RECOVERY', record2.owner_id,
    'recovery drill rescheduled after platform upgrade',
    ('CTRL-AUTHZ',), ('EV-AUTHZ-TEST',),
    'monitor authorization test and telemetry every four hours',
    'complete the recovery drill and remove the deviation', record2.owner_id,
    NOW, NOW + 5 * 86_400, NOW + 2 * 86_400, record2.system_version,
)
exception_receipt = approvals2.issue(
    'control_exception', request.payload(), record2.owner_id,
    'approver:ciso', 'ciso', NOW,
)
active_exception = registry2.activate_exception(
    request, exception_receipt.receipt_id, record2.owner_id, NOW + 1,
)
conditional = engine2.propose_decision(
    record2, evidence2, (active_exception,), NOW + 2, 'DEC-NB-CONDITIONAL',
)
print(conditional.recommendation.value, conditional.exception_ids, conditional.blockers)
assert conditional.recommendation is lab['Decision'].CONDITIONAL

## 8. Keep appetite and exceptions separate

The policy prohibits critical residual risk. Even a valid control exception cannot turn a prohibited residual band into approval.

In [ ]:
critical = replace(
    record2,
    residual=replace(
        record2.residual,
        likelihood=lab['Likelihood'].ALMOST_CERTAIN,
        impact=lab['Impact'].SEVERE,
    ),
)
critical_proposal = engine2.propose_decision(
    critical, evidence2, (active_exception,), NOW + 2, 'DEC-NB-CRITICAL',
)
print(critical_proposal.residual_band.value, critical_proposal.recommendation.value)
assert critical_proposal.recommendation is lab['Decision'].REJECT

## 9. Invalidate authorization on material change

A new model, tool, data class, trust relationship, identity provider, policy, or architecture can invalidate the causal assumptions behind the decision. Authorization must not silently survive.

In [ ]:
impacted = registry.material_change(
    record.system_id, lab['ChangeKind'].TOOL_CONTRACT, record.system_version,
    'support-v8@sha256:def', NOW + 3, 'CHG-NB-001',
)
changed = registry.records[record.risk_id]
print(impacted, changed.state.value, changed.decision_id)
assert changed.state is lab['RiskState'].REASSESSMENT_REQUIRED
assert changed.decision_id is None

## 10. Report portfolio state without false precision

Counts and rates retain their named population and denominator. Band counts stay categorical rather than being averaged into a misleading portfolio score.

In [ ]:
metrics = lab['portfolio_metrics'](
    list(registry.records.values()) + [critical],
    (active_exception,),
    NOW + 3,
)
pprint(metrics)
assert metrics['population'] == 'open risk records'
assert metrics['open_risk_count'] == 2
assert metrics['above_tolerance_count'] == 1

## 11. Inspect the decision trail

Audit events record stable identifiers, versions, outcomes, and reason codes. They deliberately avoid copying sensitive raw evidence or private chain-of-thought.

In [ ]:
for event in registry.audit:
    pprint(event)
assert {event.event_type for event in registry.audit} == {
    'risk.decision', 'risk.reassessment_required'
}

## Production boundary

Replace in-memory keys and stores with workload identity, HSM/KMS-backed signing, transactional persistence, immutable audit export, policy-as-code review, real control telemetry, and your GRC integration. Keep the invariants: exact scope, authenticated evidence, separation of duties, bounded validity, atomic consumption, explicit failure, reassessment triggers, and denominator-aware reporting.

## Reflection

1. Which facts in your own MCP deployment would invalidate an existing residual-risk assessment?
2. Which control claims currently rely on screenshots or prose rather than reproducible evidence?
3. Who may approve moderate, high, and prohibited risks, and how is separation of duties enforced?
4. What population and denominator should every executive risk metric disclose?
5. How would your system retry an unknown approval outcome without applying the decision twice?